In [0]:
from pyspark.sql.functions import *
from src.logic.transformation_functions import *

## Transformations for orders table
Clean data: (Null order_id or customer_id, Invalid order_date, Duplicate order records, Filter only valid order statuses (e.g., Completed, Shipped), Standardize date formats) 

In [0]:
# Reading orders table 
order_df = spark.read.format("delta").load("/Volumes/ecommerce_project/bronze_ecom/orders")

# display orders data
order_df.limit(10).display()


In [0]:
clean_orders = standard_date(order_df, "order_date", "cleaned_order_date")
clean_orders = clean_orders.filter(col("order_id").isNotNull())\
                        .filter(col("customer_id").isNotNull())\
                        .filter(col("cleaned_order_date").isNotNull())\
                        .dropDuplicates(["order_id"])\
                        .filter(col("order_status").isin(["Completed","Shipped"]))

display(clean_orders)


In [0]:
# writing the data
save_df_to_delta(df=clean_orders, mode="overwrite", path="ecommerce_project.silver_ecom.orders_silver")

## Transformation Customers data

Clean data: (Null customer names, Invalid city values, Duplicate customer records, Standardize customer names, Validate signup_date) 

In [0]:
customer_df = spark.read.format("delta").load("/Volumes/ecommerce_project/bronze_ecom/customers")
display(customer_df)

In [0]:
customer_silver = customer_df.filter(col("customer_name").isNotNull())\
                             .dropDuplicates(["customer_id"])\
                             .withColumn("customer_name", string_title_format("customer_name"))

customer_silver = transformed_city(customer_silver, "city")
customer_silver = standard_date(customer_silver, "signup_date", "signup_date")


In [0]:
customer_silver.display()

In [0]:
# writing the data
save_df_to_delta(df=customer_silver, mode="overwrite", path="ecommerce_project.silver_ecom.customers_silver")